**Install JDK and Spark**

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [ ]:
!wget https://dlcdn.apache.org/spark/spark-3.5.6/spark-3.5.6-bin-hadoop3.tgz
!tar xf spark-3.5.6-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.6-bin-hadoop3"

# Getting the data files
In this notebook we will use the reduced dataset (10 percent) provided for the [KDD Cup 1999](https://kdd.ics.uci.edu/databases/kddcup99/kddcup99), containing nearly half million network interactions. The file is provided as a Gzip file that we will download locally. The dataset description is available [here](https://kdd.ics.uci.edu/databases/kddcup99/kddcup.names).

In [ ]:
import requests
from pathlib import Path

url = "http://kdd.ics.uci.edu/databases/kddcup99/kddcup.data_10_percent.gz"
filename = Path("/content/kddcup.data_10_percent.gz")  # absolute path is safer in Colab

if not filename.exists():
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    filename.write_bytes(r.content)

print("File saved:", filename, "size:", filename.stat().st_size)

# Start Spark

In [ ]:
import pyspark
sc = pyspark.SparkContext('local[*]')

# Creating and RDD using parallelize
One way of creating an RDD is to parallelize an already existing list.

In [ ]:
a = range(100)
data = sc.parallelize(a)

We can `count()` the number of elements in the RDD.


In [ ]:
data.count()

We can access the first few elements on our RDD.

In [ ]:
data.take(5)

# Creating a RDD from a file
Another way of creating an RDD is to load it from a file. Notice that Spark's `textFile` can handle compressed files directly.

In [ ]:
data_file = "/content/kddcup.data_10_percent.gz"
raw_data = sc.textFile(data_file)

# Count lines
Now we have our data file loaded into the `raw_data` RDD. Without getting into Spark transformations and actions, the most basic thing we can do to check that we got our RDD contents right is to `count()` the number of lines loaded from the file into the RDD.

In [ ]:
raw_data.count()

We can also check the first few entries in our data.

In [ ]:
raw_data.take(5)

# The filter transformation
This transformation can be applied to RDDs in order to keep just elements that satisfy a certain condition. More concretely, a function is evaluated on every element in the original RDD. The new resulting RDD will contain just those elements that make the function return True. For example, imagine we want to count how many `normal.` interactions we have in our dataset. We can filter our `raw_data` RDD as follows.

In [ ]:
normal_raw_data = raw_data.filter(lambda x: 'normal.' in x)

Now we can count how many elements we have in the new RDD.

In [ ]:
from time import time

t0 = time()
normal_count = normal_raw_data.count()
tt = time() - t0

print("There are {} 'normal' interactions".format(normal_count))
print("Count completed in {} seconds".format(round(tt, 3)))

Remember that we have a total of 494021 in our 10 percent dataset. Here we can see that 97278 contain the `normal.` tag word. Notice that we have measured the elapsed time for counting the elements in the RDD. We have done this because we wanted to point out that actual computations in Spark take place when we execute *actions* and not *transformations*. In this case `count` is the action we execute on the RDD. We can apply as many transformations as we want on a our RDD and no computation will take place until we call the first action that, in this case takes a few seconds to complete.

# The map transformation
By using the `map` transformation in Spark, we can apply a function to every element in our RDD. Python's lambdas are specially expressive for this particular. In this case we want to read our data file as a CSV formatted one. We can do this by applying a lambda function to each element in the RDD as follows.

In [ ]:
from pprint import pprint

csv_data = raw_data.map(lambda x: x.split(","))

t0 = time()
head_rows = csv_data.take(5)
tt = time() - t0

print("Parse completed in {} seconds".format(round(tt,3)))
pprint(head_rows[0])

Again, all action happens once we call the first Spark *action* (i.e., `take` in this case). What if we take a lot of elements instead of just the first few?

In [ ]:
t0 = time()
head_rows = csv_data.take(100000)
tt = time() - t0

print("Parse completed in {} seconds".format(round(tt, 3)))

We can see that it takes longer. The `map` function is applied now in a distributed way to a lot of elements on the RDD, hence the longer execution time.

# Using `map` and predefined functions
Of course we can use predefined functions with `map`. Imagine we want to have each element in the RDD as a key-value pair where the key is the tag (e.g., `normal.`l) and the value is the whole list of elements that represents the row in the CSV formatted file. We could proceed as follows.

In [ ]:
def parse_interaction(line):
    elems = line.split(",")
    tag = elems[41]
    return (tag, elems)

key_csv_data = raw_data.map(parse_interaction)
head_rows = key_csv_data.take(5)
pprint(head_rows[0])

#The `collect` action
So far we have used the actions `count` and `take`. Another basic action we need to learn is `collect`. Basically it will get all the elements in the RDD into memory for us to work with them. For this reason it has to be used with care, specially when working with large RDDs. An example using our raw data.

In [ ]:
t0 = time()
all_raw_data = raw_data.collect()
tt = time() - t0

print("Data collected in {} seconds".format(round(tt, 3)))

# Sampling RDDs
In Spark, there are two sampling operations, the transformation `sample` and the action `takeSample`. By using a transformation we can tell Spark to apply successive transformation on a sample of a given RDD. By using an action we retrieve a given sample and we can have it in local memory to be used by any other standard library (e.g., Scikit-learn).
The `sample` transformation takes up to three parameters: (1) first is whether the sampling is done with replacement or not, (2) second is the sample size as a fraction, and (3) finally we can optionally provide a random seed.

In [ ]:
total_size = raw_data.count()

t0 = time()
raw_data_sample = raw_data.takeSample(False, int(0.1 * total_size), 1234)
tt = time() - t0

sample_size = len(raw_data_sample)

print("Sample size is {} of {}".format(sample_size, total_size))
print("Data collected in {} seconds".format(round(tt, 3)))

In [ ]:
total_size = raw_data.count()

t0 = time()
raw_data_sample = raw_data.sample(False, 0.1, 1234)
tt = time() - t0
sample_size = raw_data_sample.count()

print("Sample size is {} of {}".format(sample_size, total_size))
print("Data collected in {} seconds".format(round(tt, 3)))

But the power of sampling as a transformation comes from doing it as part of a sequence of additional transformations. This will show more powerful once we start doing aggregations and key-value pairs operations. In the meantime, imagine we want to have an approximation of the proportion of `normal.` interactions in our dataset. We could do this by counting the total number of tags as we did in previous notebooks. However we want a quicker response and we don't need the exact answer but just an approximation. We can do it as follows.

In [ ]:
# transformations to be applied
raw_data_sample_items = raw_data_sample.map(lambda x: x.split(","))
sample_normal_tags = raw_data_sample_items.filter(lambda x: "normal." in x)

# actions + time
t0 = time()
sample_normal_tags_count = sample_normal_tags.count()
tt = time() - t0

sample_normal_ratio = sample_normal_tags_count / float(sample_size)

print("The ratio of 'normal' interactions is {}".format(round(sample_normal_ratio,3)))
print("Count done in {} seconds".format(round(tt, 3)))

Let's compare this with calculating the ratio without sampling.

In [ ]:
# transformations to be applied
raw_data_items = raw_data.map(lambda x: x.split(","))
normal_tags = raw_data_items.filter(lambda x: "normal." in x)

# actions + time
t0 = time()
normal_tags_count = normal_tags.count()
tt = time() - t0

normal_ratio = normal_tags_count / float(total_size)

print("The ratio of 'normal' interactions is {}".format(round(normal_ratio,3)))
print("Count done in {} seconds".format(round(tt, 3)))

We can see a gain in time. The more transformations we apply after the sampling the bigger this gain. This is because without sampling all the transformations are applied to the complete set of data.

# The `takeSample` action
If what we need is to grab a sample of raw data from our RDD into local memory in order to be used by other non-Spark libraries, `takeSample` can be used. The syntax is very similar, but in this case we specify the number of items instead of the sample size as a fraction of the complete data size.

In [ ]:
t0 = time()
raw_data_sample = raw_data.takeSample(False, 400000, 1234)
normal_data_sample = [x.split(",") for x in raw_data_sample if "normal." in x]
tt = time() - t0

normal_sample_size = len(normal_data_sample)
normal_ratio = normal_sample_size / 400000.0

print("The ratio of 'normal' interactions is {}".format(normal_ratio))
print("Count done in {} seconds".format(round(tt, 3)))

The process was very similar as before. We obtained a sample of about 10 percent of the data, and then filter and split. However, it took longer, even with a slightly smaller sample. The reason is that Spark just distributed the execution of the sampling process. The filtering and splitting of the results were done locally in a single node.

# Set operations on RDDs
Spark supports many of the operations we have in mathematical sets, such as `union` and `intersection`, even when the RDDs themselves are not properly sets. It is important to note that these operations require that the RDDs being operated on are of the same type. Set operations are quite straightforward to understand as it work as expected. The only consideration comes from the fact that RDDs are not real sets, and therefore operations such as the `union` of RDDs doesn't remove duplicates. For illustrative purposes, imagine we already have our RDD with normal interactions from some previous analysis.

In [ ]:
normal_raw_data = raw_data.filter(lambda x: "normal." in x)

We can obtain `attack` (not `normal.`) interactions by subtracting `normal.` ones from the original unfiltered RDD as follows.

In [ ]:
attack_raw_data = raw_data.subtract(normal_raw_data)

Let's do some counts to check our results.

In [ ]:
# count all
t0 = time()
raw_data_count = raw_data.count()
tt = time() - t0

print("All interactions: {}".format(raw_data_count))
print("All count in {} secs".format(round(tt,3)))

In [ ]:
# count normal
t0 = time()
normal_raw_data_count = normal_raw_data.count()
tt = time() - t0

print("Normal interactions: {}".format(normal_raw_data_count))
print("Normal count in {} secs".format(round(tt,3)))

In [ ]:
# count attacks
t0 = time()
attack_raw_data_count = attack_raw_data.count()
tt = time() - t0

print("Attack interactions: {}".format(attack_raw_data_count))
print("Attack count in {} secs".format(round(tt,3)))

# Protocol and service combinations using `cartesian`
We can compute the Cartesian product between two RDDs by using the `cartesian` transformation. It returns all possible pairs of elements between two RDDs. In our case we will use it to generate all the possible combinations between service and protocol in our network interactions.
First of all we need to isolate each collection of values in two separate RDDs. For that we will use `distinct` on the CSV-parsed dataset. From the [dataset description](https://kdd.ics.uci.edu/databases/kddcup99/kddcup.names) we know that protocol is the second column and service is the third.

So first, let's get the protocols.

In [ ]:
# get the protocols
csv_data = raw_data.map(lambda x: x.split(","))
protocols = csv_data.map(lambda x: x[1]).distinct()
protocols.collect()

Now we do the same for services.

In [ ]:
# get the services
services = csv_data.map(lambda x: x[2]).distinct()
services.collect()

Now we can do the cartesian product.

In [ ]:
product = protocols.cartesian(services).collect()

print("There are {} combinations of protocol X service".format(len(product)))
print("Example combination:", product[0])

# Data aggregations on RDDs
We can aggregate RDD data in Spark by using three different actions: `reduce`, `fold`, and `aggregate`. Both `fold` and `reduce` take a function as an argument that is applied to two elements of the RDD. The `fold` action differs from `reduce` in that it gets and additional initial zero value to be used for the initial call. This value should be the identity element for the function provided. As an example, imagine we want to know the total duration of our interactions for `normal.` and `attack` interactions. We can use `reduce` as follows.

In [ ]:
csv_data = raw_data.map(lambda x: x.split(","))

# separate into different RDDs
normal_csv_data = csv_data.filter(lambda x: x[41] == "normal.")
attack_csv_data = csv_data.filter(lambda x: x[41] != "normal.")

The function that we pass to `reduce` gets and returns elements of the same type of the RDD. If we want to sum `durations` we need to extract that element into a new RDD.

In [ ]:
normal_duration_data = normal_csv_data.map(lambda x: int(x[0]))
attack_duration_data = attack_csv_data.map(lambda x: int(x[0]))

print("Normal duration data example: {}".format(normal_duration_data.take(1)))
print("Attack duration data example: {}".format(attack_duration_data.take(1)))

Now we can `reduce` these new RDDs.

In [ ]:
total_normal_duration = normal_duration_data.reduce(lambda x, y: x + y)
total_attack_duration = attack_duration_data.reduce(lambda x, y: x + y)

print("Total duration for 'normal' interactions is {}".format(total_normal_duration))
print("Total duration for 'attack' interactions is {}".format(total_attack_duration))

We can go further and use counts to calculate duration means.

In [ ]:
normal_count = normal_duration_data.count()
attack_count = attack_duration_data.count()

print("Mean duration for 'normal' interactions is {}".format(round(total_normal_duration/float(normal_count), 3)))
print("Mean duration for 'attack' interactions is {}".format(round(total_attack_duration/float(attack_count), 3)))

We have a first (and too simplistic) approach to identify attack interactions.

# A better way, using `aggregate`
The `aggregate` action frees us from the constraint of having the return be the same type as the RDD we are working on. Like with `fold`, we supply an initial zero value of the type we want to return. Then we provide two functions: (1) the first one combines the elements within each partition with the accumulator, and (2) the second one merges the accumulators coming from different partitions. This allows us to process data in parallel across partitions and then combine the partial results into a single final result. Let’s see it in action by calculating the mean as we did before.

In [ ]:
normal_sum_count = normal_duration_data.aggregate(
    (0, 0), # the initial value
    (lambda acc, value: (acc[0] + value, acc[1] + 1)), # combine value with acc
    (lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])) # combine accumulators
)

print("Mean duration for 'normal' interactions is {}".format(round(normal_sum_count[0]/float(normal_sum_count[1]), 3)))

# (0, 0) means each partition starts with (sum=0, count=0).
# For each element, update the sum and increase the count by 1.
# Example: If value=5 and acc=(10, 2), the result becomes (15, 3).
# After processing each partition, Spark combines partial results from different partitions.
# Example: (sum=30, count=3) from partition A and (sum=20, count=2) from partition B → (50, 5).
# After aggregating across all partitions, normal_sum_count will be (total_sum, total_count).
# You then compute the mean as total_sum / total_count.

In the previous aggregation, the accumulator first element keeps the total sum, while the second element keeps the count. Combining an accumulator with an RDD element consists in summing up the value and incrementing the count. Combining two accumulators requires just a pairwise sum. We can do the same with attack type interactions.

In [ ]:
attack_sum_count = attack_duration_data.aggregate(
    (0,0), # the initial value
    (lambda acc, value: (acc[0] + value, acc[1] + 1)), # combine value with acc
    (lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])) # combine accumulators
)

print("Mean duration for 'attack' interactions is {}".format(round(attack_sum_count[0]/float(attack_sum_count[1]),3)))

# Working with key/value pair RDDs
Spark provides specific functions to deal with RDDs which elements are key/value pairs. They are usually used to perform aggregations and other processings by key. Here, we want to do some exploratory data analysis on our network interactions dataset. More concretely we want to profile each network interaction type in terms of some of its variables such as duration. In order to do so, we first need to create the RDD suitable for that, where each interaction is parsed as a CSV row representing the value, and is put together with its corresponding tag as a key. Normally we create key/value pair RDDs by applying a function using `map` to the original data. This function returns the corresponding pair for a given RDD element. We can proceed as follows.

In [ ]:
csv_data = raw_data.map(lambda x: x.split(","))
key_value_data = csv_data.map(lambda x: (x[41], x)) # x[41] contains the network interaction tag

We have now our key/value pair data ready to be used. Let's get the first element in order to see how it looks like.

In [ ]:
key_value_data.take(1)

# Data aggregations with key/value pair RDDs
We can use all the transformations and actions available for normal RDDs with key/value pair RDDs. We just need to make the functions work with pair elements. Additionally, Spark provides specific functions to work with RDDs containing pair elements. They are very similar to those available for general RDDs. For example, we have a `reduceByKey` transformation that we can use as follows to calculate the total duration of each network interaction type.

In [ ]:
key_value_duration = csv_data.map(lambda x: (x[41], float(x[0])))
durations_by_key = key_value_duration.reduceByKey(lambda x, y: x + y)

durations_by_key.collect()

We have a specific counting action for key/value pairs.

In [ ]:
counts_by_key = key_value_data.countByKey()
counts_by_key

# Using `combineByKey`
This is the most general of the per-key aggregation functions. Most of the other per-key combiners are implemented using it. We can think about it as the `aggregate` equivalent since it allows the user to return values that are not the same type as our input data. For example, we can use it to calculate per-type average durations as follows.

In [ ]:
sum_counts = key_value_duration.combineByKey(
    (lambda x: (x, 1)), # the initial value, with value x and count 1
    (lambda acc, value: (acc[0]+value, acc[1]+1)), # how to combine a pair value with the accumulator: sum value, and increment count
    (lambda acc1, acc2: (acc1[0]+acc2[0], acc1[1]+acc2[1])) # combine accumulators
)

sum_counts.collectAsMap()

We can see that the arguments are pretty similar to those passed to `aggregate` in the previous notebook. The result associated to each type is in the form of a pair. If we want to actually get the averages, we need to do the division before collecting the results.

In [ ]:
duration_means_by_type = (
    sum_counts
    .map(lambda kv: (kv[0], round(kv[1][0] / kv[1][1], 3)))  # kv = (key, (sum, count))
    .collectAsMap()
)

# Print them sorted
for tag in sorted(duration_means_by_type, key=duration_means_by_type.get, reverse=True):
    print(tag, duration_means_by_type[tag])